# A big earthquake has just struck. What will the next week bring?

**EPS 88 · PyEarth.** Open your own copy on DataHub: [click here](https://datahub.berkeley.edu/hub/user-redirect/git-pull?repo=https%3A//github.com/AI4EPS/EPS88_PyEarth&branch=main&urlpath=lab/tree/EPS88_PyEarth/docs/notebooks/05_what_comes_next.ipynb).

On the morning of 4 July 2019 a magnitude 6.4 earthquake shook the desert near the town of Ridgecrest, California. For a day and a half it looked like the main event. Then, 34 hours later, a magnitude 7.1 struck close by, the largest earthquake in California in twenty years.

After a large earthquake everybody asks the same thing: is it over, or is another one coming? Nobody can predict the time, the place and the size of the next earthquake. What seismologists can give is a forecast, like a weather forecast: how many earthquakes to expect in the coming days, and the probability that one of them is large. The US Geological Survey publishes such a forecast after large earthquakes in the United States. Today you make that forecast yourself, for the week that began ten days after the magnitude 7.1, with the tools of a first course in statistics, and then check it against what the Earth did.

Every place you write something opens with a pencil icon and the words *Your turn*, and is
followed by a cell that already has the shape of the answer written in, with `...` wherever your
code goes. Replace every `...`, run every cell so your answers and figures are
saved in the file, then **download this notebook itself — the `.ipynb` file — and upload it
to Gradescope.** Not a PDF: the marking reads your notebook, and a PDF cannot be read.
In JupyterLab: **File ▸ Download**, or right-click the file in the left-hand panel and choose
**Download**.

Two habits from the first minute. A cell runs when you press **Shift+Enter**, and the notebook
remembers everything it has already run — so when something breaks and you cannot see why,
**Kernel → Restart Kernel and Run All Cells** throws the memory away and rebuilds it from the
top. That is never the wrong thing to do.

## What you'll be able to do

**The science.** Say what a large earthquake does to the earthquakes around it, how fast its
aftershocks die away, and how a forecast for the coming week is made and read.

**The statistics.** A rate · a probability as a proportion · an expected number · a simulation
and the distribution it gives · the mean and the standard deviation · the three rules for "not",
"and" and "at least one".

**The code.** A proportion in one line, `(condition).mean()` · a random draw you can repeat,
`np.random.default_rng` and `rng.choice` · a loop that lives through the same week 2,000 times ·
`.mean()` and `.std()` of the results.

**Eight places where you write something: five in class, three at home.** Each is headed
*Your turn*, and the cell under it already has the shape of the answer written in, with `...`
wherever your code goes. Replace every `...`; a comment beside each one says what it is for. A
`...` left in place is not code, and the cell will not do what you expect until it is replaced.

1. What did the mainshock change?
2. How many aftershocks will the next week bring?
3. How big will they be?
4. What is the probability of a magnitude 4 or more next week?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# house style, set once, so every plot cell below holds only what matters
plt.rcParams.update({"figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})

CACHE = "https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data"

def load(name, start, end, region):
    """Fetch one window of the USGS catalogue; fall back to the copy stored with the course.

    `region` is extra query text limiting the search to one area.
    """
    # Ask the live archive first. If it is down, or you are offline, read the copy stored with
    # the course instead, so the notebook still runs.
    try:
        return pd.read_csv(f"https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv&orderby=time-asc"
                           f"&starttime={start}&endtime={end}&minmagnitude=2.5"
                           + region)
    except Exception as e:
        print("live source unreachable, using the cached copy:", type(e).__name__)
        return pd.read_csv(CACHE + "/" + f"week05_{name}_{start}_{end}_M2.5.csv")

# Every earthquake of magnitude 2.5 or more around two California earthquakes: Ridgecrest in 2019
# for class, Loma Prieta in 1989 for the homework.
quakes = load("ridgecrest", "2019-01-01", "2020-07-01",
              "&minlatitude=35.2&maxlatitude=36.4&minlongitude=-118.2&maxlongitude=-117.0")
loma = load("lomaprieta", "1989-01-01", "1990-10-18",
            "&latitude=37.04&longitude=-121.88&maxradiuskm=60")

# One new column in each table, `days`: the time since its largest earthquake, the mainshock.
# Negative means before it. This uses dates, which you are not expected to follow yet.
for table in [quakes, loma]:
    times = pd.to_datetime(table["time"])
    mainshock = times[table["mag"].idxmax()]
    table["days"] = (times - mainshock).dt.total_seconds() / 86400

pd.set_option("display.width", 100)     # room for a table of earthquakes to print on one line
print("Ridgecrest:", quakes.shape, "   Loma Prieta:", loma.shape)

## 1. What did the mainshock change?

**What the data are.** An earthquake is a sudden slip on a fault, a break in the rock of the
Earth's crust. The slip sends out waves, and instruments called seismometers record them. From
the times at which the waves reach many seismometers, a network works out when and where the
earthquake happened and how deep it was; from the size of the waves it works out the
**magnitude**. Magnitude is a scale of size on which each step of one is about 32 times the
energy, as in the plotting week.

The US Geological Survey (USGS) lists every earthquake it locates in a public catalogue. `quakes`
is a piece of that catalogue: every event of magnitude 2.5 or more in a box about 120 km
across around Ridgecrest, from the start of 2019 to the middle of 2020. A magnitude 2.5 is
about the smallest earthquake that people nearby can feel. Each row is one event; all but two are
earthquakes, and the two are quarry blasts. Today you need four of the columns:

- `mag`, the magnitude;
- `latitude` and `longitude`, the place;
- `days`, which the setup cell added: the time since the magnitude 7.1, in days. It is
  negative for an earthquake before it, and 1.5 means a day and a half after it.

The plot below puts every row on one picture: when it happened across, how big it was up. The
print under it shows the two largest.

In [ ]:
plt.figure(figsize=(7, 3.6))
plt.scatter(quakes["days"], quakes["mag"], s=4)
plt.xlim(-190, 60)
plt.xlabel("days since the magnitude 7.1")
plt.ylabel("magnitude")
plt.title(f"Ridgecrest, six months before to two months after (n = {len(quakes)} in all)")
plt.show()

print(quakes[quakes["mag"] >= 6][["days", "mag", "place"]])       # the two largest

For six months there is a dot every week or so: the slow, ordinary activity of the region, which
seismologists call the **background**. Then comes the magnitude 6.4, the magnitude
7.1 34 hours after it, and a cloud of earthquakes that thins out as
the days pass. Together they are called a **sequence**, and its pieces have names:

- the **mainshock** is the largest earthquake of the sequence, here the magnitude 7.1;
- an **aftershock** is one of the earthquakes that follow it. A large earthquake changes the stress in the rock around it, and parts of the same fault and faults near it that were already close to failing then fail too — most within hours and days, fewer and fewer over the months that follow.
- a **foreshock** is an earthquake followed by a larger one nearby. The magnitude 6.4
  was one, and nobody could tell until the larger one came.

### ✏️ Your turn 1

**How many earthquakes a day, before the sequence and after the mainshock?**

A **rate** is a count divided by the time it took: 30 earthquakes in 10 days is a rate of 3 a
day. A rate lets you compare stretches of different length, here six quiet months and one month
of aftershocks.

Pick the two groups of rows with masks, as in the tables week:

- `before`: more than 2 days before the mainshock. The cut at 2 days keeps the
  foreshock, and the earthquakes it set off, out of the quiet period.
- `month`: the first 30 days after the mainshock.

`len` counts the rows of a table. The catalogue was searched from 186 days before
the mainshock, so the quiet period is 184 days long.

**Use these names**, because the self-check looks for them: `before`, `month`, `rate_before`,
`rate_month`.

In [ ]:
# ← your answer here: replace every ... with code
before = quakes[...]           # the rows more than 2 days before the mainshock: days below -2
month = quakes[(...) & (...)]  # the first 30 days after it: days above 0, and days below 30
rate_before = len(...) / 184   # earthquakes a day, over the 184 days before
rate_month = len(...) / 30     # earthquakes a day, over the first 30 days

print("before:", len(before), "earthquakes in 184 days,", round(rate_before, 2), "a day")
print("first month:", len(month), "earthquakes in 30 days,", round(rate_month), "a day")
print("the rate went up", round(rate_month / rate_before), "times")

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert (before["days"] < -2).all() and len(before) == (quakes["days"] < -2).sum(), "before keeps every row whose days is below -2, and no other"
assert (month["days"] > 0).all() and (month["days"] < 30).all() and len(month) == ((quakes["days"] > 0) & (quakes["days"] < 30)).sum(), "month keeps every row with days above 0 and below 30, and no other: two conditions joined with &"
assert abs(rate_before - len(before) / 184) < 1e-9 and abs(rate_month - len(month) / 30) < 1e-9, "each rate is the number of rows, len(...), divided by the number of days"
print(f"✓ Your turn 1 — {round(rate_before, 2)} earthquakes a day before, {round(rate_month)} a day in the first month")

One earthquake a week became 73 a day: the rate went up more than five
hundred times. The map shows where they were, with your `before` in grey and everything after the
mainshock in red.

In [ ]:
after = quakes[quakes["days"] > 0]                # every row after the mainshock

plt.figure(figsize=(6, 5.4))
plt.scatter(after["longitude"], after["latitude"], s=2, color="crimson", label="after the mainshock")
plt.scatter(before["longitude"], before["latitude"], s=14, color="0.35", label="in the six months before")
plt.scatter(-117.6709, 35.6225, s=60, marker="s", color="black")
plt.text(-117.6709 - 0.33, 35.6225 - 0.02, "Ridgecrest")
plt.gca().set_aspect("equal")
plt.xlabel("longitude (degrees east)")
plt.ylabel("latitude (degrees north)")
plt.title(f"Before (n = {len(before)}) and after (n = {len(after)})")
plt.legend(loc="lower left")
plt.show()

The aftershocks are not scattered over the box. Most of them draw one band about 50 km long,
running from north-west to south-east past the town: the fault that broke. **Aftershocks happen
on and around the fault that slipped in the mainshock.**

So how many earthquakes to expect here in a day is not a fixed number: it depends on what has
just happened, and where. To forecast the days ahead you need to know how fast the rate is
falling.

## 2. How many aftershocks will the next week bring?

At 73 a day against one a week, nearly every earthquake in the box is now an
aftershock, so from here on count them all as aftershocks.

Put yourself ten days after the mainshock: you know everything up to now and nothing after. First
see how the aftershocks are spread over those ten days. A histogram cuts the time into bins, here
one bin for each day, and counts the aftershocks that fall in each. `plt.hist` draws it, and
`np.histogram` with the same bins gives the counts as numbers. Call the first 24 hours day 1, the
next 24 hours day 2, and so on.

In [ ]:
first_ten = quakes[(quakes["days"] > 0) & (quakes["days"] < 10)]

plt.figure(figsize=(7, 3.6))
plt.hist(first_ten["days"], bins=np.arange(0, 11))           # one bin for each day
plt.xlabel("days since the mainshock")
plt.ylabel("aftershocks in the day")
plt.title(f"The first ten days (n = {len(first_ten)} aftershocks)")
plt.show()

daily, edges = np.histogram(first_ten["days"], bins=np.arange(0, 11))
day = np.arange(1, 11)                                        # day 1 is the first 24 hours, day 2 the next, ...
print("aftershocks on days 1 to 10:", daily)

The count falls fast at first and then more and more slowly: 1257 on day 1,
170 on day 2, 76 on day 4, 29 on day 8. From day 2 on, each
time the day number doubles, the count roughly halves.

**Omori's law.** The number of aftershocks in a day falls roughly as one over the time since the mainshock: twice as late, about half as many. Fusakichi Omori found it in Japan in 1894. As a formula:

***count on a day = K / day***

A formula like this, which stands in for the data, is called a **model**. K is the one number in
it that can be adjusted. It belongs to this sequence: a larger K means more aftershocks on every
day, and a sequence after a larger mainshock has a larger K. Once K is known the model is a
forecast, because you can put in a day that has not happened yet.

### ✏️ Your turn 2

**Which K fits this sequence, and how many aftershocks should the next seven days bring?**

Find K by trying. The cell draws the histogram again with the curve K / day on top. The axis is a
log axis, as in the plotting week, because the counts run from more than a thousand down to
thirty and an ordinary axis would squash the small ones flat. Start with `K = 100`, run the cell,
and change K until the curve follows the tops of the bars from day 2 on. Day 1 stays far above
the curve: in the first hours the count falls too fast for so simple a formula. (How a computer
finds the best value for you is the subject of the regression week; today your eye is the
judge.)

Then use your K. `next_day` holds the days of the coming week, 11 to 17. K
divided by it is the count to expect on each of those days, and their sum is the forecast for the
week.

**Use these names**, because the self-check looks for them: `K`, `rates`, `expected`.

In [ ]:
# ← your answer here: replace every ... with code
K = ...  # try 100 first, then change it until the curve follows the bars from day 2 on
next_day = np.arange(11, 18)

plt.figure(figsize=(7, 3.6))
plt.hist(first_ten["days"], bins=np.arange(0, 11))
plt.plot(..., ..., color="firebrick")  # the curve: the day numbers across, K over the day number up
plt.yscale("log")                      # a log axis keeps the small counts readable
plt.xlabel("days since the mainshock")
plt.ylabel("aftershocks in the day")
plt.title(f"Omori's law with K = {K}")
plt.show()

rates = ... / ...       # the count to expect on each of days 11 to 17: K over the day number
expected = (...).sum()  # added up over the seven days

print("K =", K)
print("expected on each of days 11 to 17:", rates.round())
print("expected in days 11 to 17:", round(expected, 1))

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert 250 <= K <= 400, "choose K so that the curve follows the tops of the bars from day 2 on: a value between about 250 and 400"
assert len(rates) == 7 and abs(rates[0] - K / next_day[0]) < 1e-9, "rates is K divided by next_day: one expected count for each of the seven days"
assert abs(expected - rates.sum()) < 1e-9, "expected is the seven rates added up"
print(f"✓ Your turn 2 — with K = {K}, about {round(expected)} aftershocks are expected in days 11 to 17")

A K near 320 follows the bars from day 2 on. With it the coming week should bring about
29 aftershocks on day 11, falling to 19 on day
17: **163** in all. Your K may differ a little, and so may your total. So that
everyone's numbers agree from here on, the notebook takes the week's count as 163.

## 3. How big will they be?

About 163 aftershocks are coming. Most will be too small to matter. What people want to know
is whether a larger one is among them. So the next question is: what is the probability that an
aftershock is large?

**Probability.** The proportion of many tries in which a thing happens. It is a number between 0 and 1: 0 means never, 1 means
always, 0.5 means half the time. A weather forecast of "30% rain" says that on 3 of every 10 days
like this one, it rains.

The tries here are the aftershocks already recorded. `sample` holds the aftershocks from one day
after the mainshock to ten days after it, which is days 2 to 10. Both limits have a reason:

- **It stops at ten days** because that is now. A forecast can use only what has already
  happened.
- **It leaves out day 1** because the list for that day is not complete. In the hours after a
  large earthquake the ground does not stop shaking: the waves of one aftershock run into the
  waves of the next, and only the larger ones can be picked out. Of the first ten aftershocks in
  the catalogue, 9 are magnitude 4 or more. Counting that day would make large
  aftershocks look more common than they are.

The cell below draws the magnitudes of the sample, and then works out one proportion by hand, on
ten of them.

In [ ]:
sample = quakes[(quakes["days"] >= 1) & (quakes["days"] < 10)]   # from 1 day to 10 days after the mainshock
n = 163                                                          # aftershocks expected in days 11 to 17, with K = 320

plt.figure(figsize=(9, 3.6))
plt.subplot(1, 2, 1)
plt.hist(sample["mag"], bins=np.arange(2.5, 5.2, 0.1))
plt.axvline(4, color="firebrick")
plt.xlabel("magnitude")
plt.ylabel("number of aftershocks")
plt.title("ordinary axis")

plt.subplot(1, 2, 2)
plt.hist(sample["mag"], bins=np.arange(2.5, 5.2, 0.1))
plt.axvline(4, color="firebrick")
plt.yscale("log")
plt.xlabel("magnitude")
plt.title(f"log axis (n = {len(sample)} aftershocks)")
plt.show()

ten = np.array(sample["mag"])[:10]                # the first ten magnitudes of the sample
print("ten magnitudes:    ", ten)
print("3 or more?         ", ten >= 3)
print("how many are True: ", (ten >= 3).sum(), "of 10")
print("the proportion:    ", (ten >= 3).mean())

Small aftershocks far outnumber large ones: the bars fall away quickly to the right, and on the
log axis they fall along a rough straight line, as the eruptions did. The red line marks
magnitude 4, large enough to be felt well beyond the place where it happens.

The printout shows how a proportion is worked out. `ten >= 3` is a mask: it asks "is this one
magnitude 3 or more?" of every value and answers True or False. 5 of the ten are
True, a proportion of 0.5. `.mean()` of a mask gives that proportion in one step,
because Python counts True as 1 and False as 0.

Ten aftershocks are too few to trust. The whole sample has 637, and its proportion is
the best measure there is of the probability for an aftershock that has not happened yet. That
rests on one assumption: next week's aftershocks will be like the ones seen so far. Aftershocks do
become fewer as the days pass, but they do not become smaller (look back at the first figure), so
the assumption is a fair one.

**Expected number.** How many to expect on average: the number of tries times the probability of each. With 10 tries and a probability of 0.5, expect
10 × 0.5 = 5. It is an average over many repeats, so it does not have to be a whole number.

### ✏️ Your turn 3

**What is the probability that an aftershock is magnitude 3 or more, and magnitude 4 or more, and
how many of each should next week bring?**

Work out the two proportions on the whole sample, as the cell above did on ten values. Then the
two expected numbers: the tries are the `n` aftershocks of next week.

**Use these names**, because the self-check looks for them: `p3`, `p4`, `expected_3`,
`expected_4`.

In [ ]:
# ← your answer here: replace every ... with code
p3 = (...).mean()       # the proportion of the sample that is magnitude 3 or more
p4 = (...).mean()       # the proportion that is magnitude 4 or more
expected_3 = ... * ...  # how many of the n aftershocks to expect at magnitude 3 or more
expected_4 = ... * ...  # and at magnitude 4 or more

print((sample["mag"] >= 3).sum(), "of", len(sample), "are magnitude 3 or more: a proportion of", round(p3, 3))
print((sample["mag"] >= 4).sum(), "of", len(sample), "are magnitude 4 or more: a proportion of", round(p4, 3))
print("expected next week:", round(expected_3), "of magnitude 3 or more,", round(expected_4, 1), "of magnitude 4 or more")

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert abs(p3 - (sample["mag"] >= 3).mean()) < 1e-9 and 0.30 < p3 < 0.37, "p3 is the proportion of the sample with magnitude 3 or more: the mean of a mask on sample, about a third"
assert abs(p4 - (sample["mag"] >= 4).mean()) < 1e-9 and 0.01 < p4 < 0.03, "p4 is the proportion of the sample with magnitude 4 or more, about 0.02"
assert abs(expected_3 - n * p3) < 1e-9 and abs(expected_4 - n * p4) < 1e-9, "an expected number is the number of tries, n, times the probability"
print(f"✓ Your turn 3 — a probability of {round(p3, 3)} for magnitude 3 or more and {round(p4, 3)} for 4 or more: expect {round(expected_3)} and {round(expected_4, 1)} next week")

A third of the aftershocks are magnitude 3 or more, and about one in fifty is magnitude 4 or more
(13 of 637). So of next week's 163, expect about 54 of magnitude 3 or
more and about 3.3 of magnitude 4 or more.

## 4. What is the probability of a magnitude 4 or more next week?

Expect about 3.3 of them. But an expected number is an average, and the week that comes
will hold a whole number: 0, 1, 2, 3 or more. What people ask is whether it will be 0. To find the
probability of each outcome, let the computer live through next week many times and count. That
is a **simulation**.

### One possible week

A possible week is 163 aftershocks, each with a magnitude drawn at random from the sample.
`np.random.default_rng(seed)` makes a random-number generator. A computer's random numbers come from a recipe, and the seed is where the recipe starts, so the same seed gives the same draws every time.
`rng.choice(values, n)` draws `n` of the values at random; the same value may be drawn more than
once.

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
sample = quakes[(quakes["days"] >= 1) & (quakes["days"] < 10)]   # from 1 day to 10 days after the mainshock
n = 163                                                          # aftershocks expected in days 11 to 17, with K = 320
p3 = 212 / 637                                                   # the probabilities from Your turn 3
p4 = 13 / 637
first_ten = quakes[(quakes["days"] > 0) & (quakes["days"] < 10)]
all_days = np.arange(1, 18)                                   # days 1 to 17

In [ ]:
rng = np.random.default_rng(1)

five = rng.choice(sample["mag"], 5)               # five magnitudes drawn at random from the sample
print("five magnitudes:", five)

one_week = rng.choice(sample["mag"], n)           # one possible week: n magnitudes
print("one possible week:", (one_week >= 4).sum(), "of magnitude 4 or more")

another_week = rng.choice(sample["mag"], n)       # draw again, and the week is a different one
print("another possible week:", (another_week >= 4).sum(), "of magnitude 4 or more")

`five` shows what a draw is: sizes picked from the sample. `one_week` and `another_week` are each
a whole possible week of 163 sizes, and they do not hold the same number of magnitude 4 or
more. One week is an anecdote; two thousand are a forecast.

### ✏️ Your turn 4

**What do 2,000 possible weeks look like?**

The loop does what the cell above did, 2,000 times: it draws a week, counts its aftershocks of
magnitude 4 or more, and keeps that one number in the list `larger`. After the loop, `np.array`
turns the list into an array, so that a mask and `.mean()` work on it. Then describe the
2,000 numbers with three more:

- the proportion of the weeks that hold at least one, which is the probability you are after;
- their **mean**, `.mean()`, the average number in a week;
- their **standard deviation**, `.std()`, which says how spread out they are.

**Standard deviation.** How far a value usually is from the average. A small one means the weeks are much alike;
a large one means they differ a lot.

**Use these names**, because the self-check looks for them: `larger`, `prob_sim`, `mean_larger`,
`std_larger`.

In [ ]:
# ← your answer here: replace every ... with code
rng = np.random.default_rng(88)
larger = []
for run in range(2000):
    week = rng.choice(..., ...)  # a possible week: n magnitudes drawn at random from the sample
    larger.append(...)           # how many of the week's magnitudes are 4 or more
larger = np.array(larger)

prob_sim = (...).mean()  # the proportion of the weeks with at least one
mean_larger = ...        # the mean of larger
std_larger = ...         # the standard deviation of larger

print("the first ten weeks:", larger[:10])
print("at least one in", round(prob_sim, 3), "of the weeks")
print("mean", round(mean_larger, 1), "  standard deviation", round(std_larger, 1))

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert len(larger) == 2000 and larger.max() < 40, "larger holds one number for each of the 2000 weeks: how many of that week's aftershocks are magnitude 4 or more"
assert 3.08 < larger.mean() < 3.57 and 1.53 < larger.std() < 2.08, "each week is n magnitudes drawn from the sample's mag column, and larger keeps the COUNT of those that are 4 or more: about 3.3 on average"
assert abs(prob_sim - (larger >= 1).mean()) < 1e-9, "prob_sim is the proportion of weeks where larger is 1 or more: the mean of a mask"
assert abs(mean_larger - larger.mean()) < 1e-9 and abs(std_larger - larger.std()) < 1e-9, "mean_larger is larger.mean() and std_larger is larger.std()"
print(f"✓ Your turn 4 — at least one in {round(100 * prob_sim)}% of 2000 weeks; mean {round(mean_larger, 1)}, standard deviation {round(std_larger, 1)}")

In [ ]:
plt.figure(figsize=(7, 3.6))
plt.hist(larger, bins=np.arange(0, larger.max() + 2), align="left")       # one bar for each whole number
plt.axvline(mean_larger, color="firebrick")                                 # the mean
plt.axvline(mean_larger - std_larger, color="firebrick", ls="--")           # one standard deviation below it
plt.axvline(mean_larger + std_larger, color="firebrick", ls="--")           # and one above it
plt.xlabel("aftershocks of magnitude 4 or more in the week")
plt.ylabel("number of weeks")
plt.title("2,000 possible weeks (n = 2,000)")
plt.show()

inside = (larger > mean_larger - std_larger) & (larger < mean_larger + std_larger)
print("weeks within one standard deviation of the mean:", round(inside.mean(), 2))
for tries in [10, 100, 1000, 2000]:
    print("after", tries, "weeks, the proportion with at least one:", round((larger[:tries] >= 1).mean(), 2))

### What the 2,000 weeks say

The histogram is a **distribution**: it shows how often each outcome came up. Your three numbers
describe it.

- **The probability of at least one** is 0.962: that proportion of the weeks hold
  one or more. Only the first bar, the weeks with none, is left out.
- **The mean** is 3.3, the solid line. It is the expected number of Your turn 3, found
  a second way.
- **The standard deviation** is 1.8. The dashed lines stand one standard deviation
  below and above the mean, at 1.5 and 5.1, and 72% of the weeks
  fall between them. So "about 3.3" really means "usually between 2 and
  5".

The last lines of the printout show why it takes many tries. After 10 weeks the proportion reads
1; by 100 it is near 0.96, and it stays there. A probability is
the value the proportion settles on. Working one out by many random tries is also called the
**Monte Carlo** method. Let the computer live through the same situation thousands of times, each time with fresh random draws, and count how often a thing happens.

### The rules of probability behind it

The same probability can be worked out with three rules and no simulation.

**Rule 1: not.** The probability that a thing does *not* happen is 1 minus the probability that
it does. One aftershock is magnitude 4 or more with probability 0.02, so it is smaller
with probability 1 − 0.02 = 0.98.

**Rule 2: and.** If two things are independent, the probability that *both* happen is the product
of their two probabilities. A coin lands tails with probability 0.5, so two coins both land tails
with probability 0.5 × 0.5 = 0.25: one time in four.

**Independent.** Knowing how one turned out tells you nothing about the next. Two coins are independent. Two aftershocks nearly are:
the size of one says almost nothing about the size of the next.

Two aftershocks are both smaller than magnitude 4 with probability
0.98 × 0.98 = 0.96. Three are, with probability
0.98 × 0.98 × 0.98 = 0.94. And `n` of them
are, with 0.98 multiplied by itself `n` times, which Python writes `0.98 ** n`.

**Rule 3: at least one.** "At least one is magnitude 4 or more" is the opposite of "all of them
are smaller". So by rule 1 its probability is 1 minus the probability from rule 2.

The cell below uses the three rules for every number of aftershocks from 1 to 300.

In [ ]:
counts = np.arange(1, 301)                        # 1, 2, 3, ... 300 aftershocks
all_smaller = (1 - p4) ** counts                  # rules 1 and 2: every one of them is below magnitude 4
at_least_one = 1 - all_smaller                    # rule 3

plt.figure(figsize=(7, 3.6))
plt.plot(counts, at_least_one)
plt.xlabel("number of aftershocks")
plt.ylabel("probability of at least one\nof magnitude 4 or more")
plt.title("Many small probabilities add up")
plt.show()

print("with 1 aftershock:", round(at_least_one[0], 2))
print("with 10:", round(at_least_one[9], 2))
print("with 50:", round(at_least_one[49], 2))
print("with 100:", round(at_least_one[99], 2))

One aftershock is magnitude 4 or more with probability 0.02. Among 10 there is at
least one with probability 0.19, among 50 with 0.64, among 100 with
0.87. Each aftershock is unlikely to be large, but there are many of them, and that is
the whole logic of an aftershock forecast.

### ✏️ Your turn 5

**What is the probability of at least one aftershock of magnitude 4 or more tomorrow, and in the
whole week?**

Use the three rules. The week should bring `n` aftershocks. Tomorrow, day 11, should bring
about 29 of them, which the cell calls `n_tomorrow`.

**Use these names**, because the self-check looks for them: `p_small`, `prob_none`, `prob_week`,
`prob_tomorrow`.

In [ ]:
# ← your answer here: replace every ... with code
n_tomorrow = 29

p_small = 1 - ...               # rule 1: the probability that one aftershock is below magnitude 4
prob_none = ... ** ...          # rule 2: the probability that all n of the week's aftershocks are
prob_week = 1 - ...             # rule 3: the probability of at least one in the week
prob_tomorrow = 1 - ... ** ...  # rules 2 and 3 in one line, for tomorrow's aftershocks

print("the week: none with probability", round(prob_none, 3), "  at least one with probability", round(prob_week, 3))
print("tomorrow: at least one with probability", round(prob_tomorrow, 3))

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert abs(p_small - (1 - p4)) < 1e-9 and abs(prob_none - (1 - p4) ** n) < 1e-9, "p_small is 1 minus p4, and prob_none is p_small raised to the power n"
assert abs(prob_week - (1 - (1 - p4) ** n)) < 1e-9, "prob_week is 1 minus prob_none"
assert abs(prob_tomorrow - (1 - (1 - p4) ** n_tomorrow)) < 1e-9, "prob_tomorrow is 1 minus p_small raised to the power n_tomorrow"
print(f"✓ Your turn 5 — at least one of magnitude 4 or more: {round(100 * prob_tomorrow)}% tomorrow, {round(100 * prob_week)}% in the week")

The rules give 97% for the week and the simulation gave 96%:
the same answer, because both rest on the same things: that exactly 163 aftershocks come,
that 0.02 is the true probability, and that the sizes are independent. Tomorrow alone is
close to an even chance, 45%; over a week it is nearly certain. The rules
are quicker. The simulation also gives the mean and the standard deviation, and it answers
questions these rules do not, such as the probability of two or more, which the homework asks.

So the forecast for days 11 to 17 reads: about 163 aftershocks, about
54 of them magnitude 3 or more, and a 97% probability of at least one of
magnitude 4 or more, usually between 2 and 5 of them. Now look at what the
Earth did.

In [ ]:
came = quakes[(quakes["days"] >= 10) & (quakes["days"] < 17)]
came_3 = (came["mag"] >= 3).sum()
came_4 = (came["mag"] >= 4).sum()

print("aftershocks in days 11 to 17:", len(came), "   expected:", n)
print("of magnitude 3 or more:", came_3, "   expected:", round(n * p3))
print("of magnitude 4 or more:", came_4, "   expected:", round(n * p4, 1))
print("the largest:", came["mag"].max())

plt.figure(figsize=(7, 3.6))
plt.hist(first_ten["days"], bins=np.arange(0, 11), label="counted, days 1 to 10")
plt.hist(came["days"], bins=np.arange(10, 18), color="darkorange", label="what came, days 11 to 17")
plt.plot(all_days, 320 / all_days, color="firebrick", label="the forecast: K / day with K = 320")
plt.yscale("log")
plt.xlabel("days since the mainshock")
plt.ylabel("aftershocks in the day")
plt.title(f"The forecast and what came (n = {len(came)} aftershocks in days 11 to 17)")
plt.legend()
plt.show()

169 aftershocks came where you expected 163, 62 of magnitude 3 or more
where you expected 54, and 3 of magnitude 4 or more where you expected
3.3 with a standard deviation of 1.8.

The US Geological Survey published its own forecast on 15 July 2019, for nearly the same week:
between 23 and 52 aftershocks of magnitude 3 or more, with a probability above 99% of at least
one. Your 54 is just above the top of that range. The Survey's counts are not taken over
exactly this box, so the two numbers are not exactly like for like. Its forecast is built from the same two
ingredients as yours: how fast the count falls, and what proportion of aftershocks are large.

Two limits of what you built:

- It is rough. The curve runs above some bars and below others, and what came was above the
  forecast at first and below it at the end. The Survey renews its forecast as the sequence goes
  on.
- The Survey also gave a 30% probability of a magnitude 5 or more. Your sample holds none (its
  largest is 4.9), so its proportion is 0, and that cannot be right. A proportion
  cannot be measured for a size the sample does not hold. How to reach sizes too rare to be in
  the sample is the subject of the transformations week.

And one aftershock was larger than all of these: a magnitude 5.5, about
11 months after the mainshock.

## The question, answered

Ten days after the Ridgecrest mainshock about 160 more aftershocks were due within
a week and one in fifty aftershocks was magnitude 4 or more, so at least one more of that size
was nearly certain, and 3 came.

## Week 5 summary

### The science

**A big earthquake has just struck. What will the next week bring?**  
A large earthquake raises the probability of more earthquakes around it, and that extra rate dies away roughly as one over the time since it. A probability is a proportion of many tries: of what has happened, or of futures you simulate. A forecast is a count and a proportion turned into an expected number and a probability of at least one, and the week that comes can still differ from it.

- **Probability.** The proportion of many tries in which a thing happens.
- **Omori's law.** The number of aftershocks in a day falls roughly as one over the time since the mainshock: twice as late, about half as many.
- **Expected number.** How many to expect on average: the number of tries times the probability of each.
- **Monte Carlo.** Let the computer live through the same situation thousands of times, each time with fresh random draws, and count how often a thing happens.
- **Standard deviation.** How far a value usually is from the average.
- **Independent.** Knowing how one turned out tells you nothing about the next.
- **Aftershock sequence.** A large earthquake changes the stress in the rock around it, and parts of the same fault and faults near it that were already close to failing then fail too — most within hours and days, fewer and fewer over the months that follow.

### The code

| Function | What it does |
|---|---|
| **Python** | |
| `len(table)` | how many rows |
| `for x in things:` | do the same thing once for each item |
| `range(n)` | the numbers 0 to n-1, to count with |
| `list.append(x)` | add one item to the end of a list |
| `round(x, 1)` | trim a long decimal — 9.1 rather than sixteen digits |
| **NumPy** | |
| `array.std()` | the standard deviation: how far a value usually is from the average |
| `np.random.default_rng(seed)` | a random-number generator you can reproduce — the same seed gives the same draws |
| `rng.choice(values, n)` | n of the values drawn at random; the same one may be drawn more than once |
| `np.array(list)` | the container that does arithmetic to every number at once |
| **pandas** | |
| `mask.mean()` | the proportion of a mask that is True, in one line |
| `series.mean()` | the average value |
| `table[mask]` | keep only the rows where the mask is True |
| **Matplotlib** | |
| `plt.plot(x, y)` | join points with a line — here, the model K / day drawn over the histogram |

## Homework

Three parts on the Bay Area's own earthquake. At 5:04 in the afternoon of 17 October 1989 the
magnitude 6.9 Loma Prieta earthquake struck the Santa Cruz Mountains, about 100 km south of
campus. It is the earthquake that interrupted the 1989 World Series and broke a section of the
Bay Bridge. `loma` holds every event of magnitude 2.5 or more within 60 km of it, from the
start of 1989 to a year after the mainshock: 644 rows, with the same `days` column,
counted from its own mainshock. The first cell below shows the sequence.

The three parts make and test a forecast for Loma Prieta, as class did for Ridgecrest: the model
K / day and the three rules in part 1, the rules again in part 2, the simulation loop in part 3.
If you have restarted the kernel, run the setup cell at the top and the checkpoint below; nothing
else is needed.

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
sample = quakes[(quakes["days"] >= 1) & (quakes["days"] < 10)]   # from 1 day to 10 days after the mainshock
n = 163                                                          # aftershocks expected in days 11 to 17, with K = 320

In [ ]:
plt.figure(figsize=(7, 3.6))
plt.scatter(loma["days"], loma["mag"], s=5)
plt.xlim(-60, 60)
plt.xlabel("days since the magnitude 6.9")
plt.ylabel("magnitude")
plt.title("Loma Prieta: every event from two months before to two months after")
plt.show()

### ✏️ Your turn 6

**Ten days after Loma Prieta, what was the probability of an aftershock of magnitude 4 or more in the
next week?**

Make the forecast as class did, from the first ten days alone. Find K by trying values until the
curve follows the bars from day 2 on; this fit is rougher than Ridgecrest's, because there are
fewer aftershocks. Then work out the count to expect in days 11 to 17,
rounded to a whole number. Then take the sample from one day to ten days after the mainshock, as
class did and for the same two reasons, find the proportion of it that is magnitude 4 or more,
and use the three rules for the probability of at least one. The last lines print what came and
draw your forecast against it.

**Use these names**, because the self-check looks for them: `loma_ten`, `K_loma`,
`n_loma`, `sample_loma`, `p_loma`, `prob_loma`.

In [ ]:
# ← your answer here: replace every ... with code
loma_ten = loma[(...) & (...)]  # the first ten days after the mainshock: days above 0, and days below 10
K_loma = ...                    # try 20 first, then change it until the curve follows the bars from day 2 on
day = np.arange(1, 11)
next_day = np.arange(11, 18)
all_days = np.arange(1, 18)

plt.figure(figsize=(7, 3.6))
plt.hist(loma_ten["days"], bins=np.arange(0, 11))
plt.plot(..., ..., color="firebrick")  # the curve: the day numbers across, K_loma over the day number up
plt.yscale("log")
plt.xlabel("days since the mainshock")
plt.ylabel("aftershocks in the day")
plt.title(f"Loma Prieta: Omori's law with K = {K_loma}")
plt.show()

n_loma = round((... / ...).sum())  # the count to expect in days 11 to 17, as a whole number
sample_loma = loma[(...) & (...)]  # the aftershocks of days 2 to 10: the days column at least 1, and below 10
p_loma = (...).mean()              # the proportion of that sample that is magnitude 4 or more
prob_loma = 1 - (1 - ...) ** ...   # the probability of at least one among the n_loma

came_loma = loma[(loma["days"] >= 10) & (loma["days"] < 17)]
print("K =", round(K_loma), "  ", n_loma, "expected in days 11 to 17")
print((sample_loma["mag"] >= 4).sum(), "of", len(sample_loma), "in the sample are magnitude 4 or more: a proportion of", round(p_loma, 2))
print(f"the probability of at least one of magnitude 4 or more: {round(100 * prob_loma)}%")
print("came:", len(came_loma), "aftershocks,", (came_loma["mag"] >= 4).sum(), "of magnitude 4 or more")

plt.figure(figsize=(7, 3.6))
plt.hist(loma_ten["days"], bins=np.arange(0, 11), label="counted, days 1 to 10")
plt.hist(came_loma["days"], bins=np.arange(10, 18), color="darkorange", label="what came, days 11 to 17")
plt.plot(all_days, K_loma / all_days, color="firebrick", label="your forecast: K / day")
plt.yscale("log")
plt.xlabel("days since the mainshock")
plt.ylabel("aftershocks in the day")
plt.title("Loma Prieta: the forecast and what came")
plt.legend()
plt.show()

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert (loma_ten["days"] > 0).all() and (loma_ten["days"] < 10).all() and len(loma_ten) > 300, "loma_ten keeps the rows of loma with days above 0 and below 10"
assert 25 <= K_loma <= 70, "choose K_loma so that the curve follows the tops of the bars from day 2 on: a value between about 25 and 70"
assert n_loma == round((K_loma / next_day).sum()), "n_loma is K_loma over next_day, added up and rounded"
assert len(sample_loma) == ((loma["days"] >= 1) & (loma["days"] < 10)).sum() and (sample_loma["days"] >= 1).all() and abs(p_loma - (sample_loma["mag"] >= 4).mean()) < 1e-9, "sample_loma keeps every row of loma from 1 day to 10 days after the mainshock, and p_loma is its proportion of magnitude 4 or more"
assert abs(prob_loma - (1 - (1 - p_loma) ** n_loma)) < 1e-9, "prob_loma is 1 minus the probability that all n_loma are below magnitude 4"
print(f"✓ Homework 1 — {n_loma} aftershocks expected, a proportion of {round(p_loma, 2)} at magnitude 4 or more, so a {round(100 * prob_loma)}% probability of at least one")

**What the numbers mean.** Compare your forecast with the last line of your printout. Did the count come close, and did the
likely thing happen? Then think about what the proportion rests on: how many earthquakes of magnitude 4
or more are in your sample? Print them with `sample_loma[sample_loma["mag"] >= 4]["mag"]` if you like.
They are listed to one decimal. What would happen to your proportion if the question had
been "more than 4" and not "4 or more"?

### ✏️ Your turn 7

**Whose proportion should the forecast use: Loma Prieta's own, or Ridgecrest's?**

Ten days into a sequence a forecaster has few large aftershocks to measure a proportion from:
here 12, out of 107. Another earthquake offers a second measurement: Ridgecrest's
sample, `sample`, with 13 out of 637. Work out Ridgecrest's proportion again, then the probability
of at least one and the expected number with each proportion. Part 1's two results are written in, so
this part stands on its own.

**Use these names**, because the self-check looks for them: `p_ridge`, `prob_own`,
`prob_borrowed`, `expected_own`, `expected_borrowed`.

In [ ]:
# ← your answer here: replace every ... with code
n_loma = 24        # from part 1: the aftershocks expected in days 11 to 17
p_loma = 12 / 107  # from part 1: Loma Prieta's own proportion

p_ridge = (...).mean()                # the proportion of Ridgecrest's sample that is magnitude 4 or more
prob_own = 1 - (1 - ...) ** ...       # at least one among the n_loma, with Loma Prieta's own proportion
prob_borrowed = 1 - (1 - ...) ** ...  # the same with Ridgecrest's proportion
expected_own = ... * ...              # the number to expect with Loma Prieta's own proportion
expected_borrowed = ... * ...         # and with Ridgecrest's

print(f"Loma Prieta's own proportion, {round(p_loma, 3)}: a {round(100 * prob_own)}% probability, {round(expected_own, 1)} expected")
print(f"Ridgecrest's proportion, {round(p_ridge, 3)}: a {round(100 * prob_borrowed)}% probability, {round(expected_borrowed, 1)} expected")

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert abs(p_ridge - (sample["mag"] >= 4).mean()) < 1e-9, "p_ridge is the proportion of sample, Ridgecrest's aftershocks, that is magnitude 4 or more"
assert abs(prob_own - (1 - (1 - p_loma) ** n_loma)) < 1e-9 and abs(prob_borrowed - (1 - (1 - p_ridge) ** n_loma)) < 1e-9, "each probability is 1 minus the probability that all n_loma are below magnitude 4: p_loma in one, p_ridge in the other"
assert abs(expected_own - n_loma * p_loma) < 1e-9 and abs(expected_borrowed - n_loma * p_ridge) < 1e-9, "an expected number is the number of tries, n_loma, times the proportion"
print(f"✓ Homework 2 — a {round(100 * prob_own)}% probability with Loma Prieta's own proportion, {round(100 * prob_borrowed)}% with Ridgecrest's")

**What the numbers mean.** Your printout holds two forecasts made with the same method. How far apart are they, and which
would you have published on day 10? Think about what each proportion rests on: Loma Prieta's own is
about the right earthquake, but how many aftershocks of magnitude 4 or more stand behind it?
How many stand behind Ridgecrest's? It is also a different fault, thirty years later, recorded by
different instruments. What extra data would tell you which proportion was nearer the truth?

### ✏️ Your turn 8

**2 aftershocks of magnitude 4 or more came. Which forecast does that favour, and what
did the year that followed say?**

Simulate the week 2,000 times with each sample, as Your turn 4 did with one. This time the loop is
yours to write: the generator, the `for` line and what each `append` keeps. `own` keeps the number
of magnitude 4 or more when the 24 sizes are drawn from Loma Prieta's sample, `borrowed` when
they are drawn from Ridgecrest's. Then find, for each, the proportion of weeks with 2 or
more. Last, measure the proportion directly from what a forecaster on day 10 could not see: the
earthquakes after day 17.

**Use these names**, because the self-check looks for them: `sample_loma`, `own`, `borrowed`,
`prop_own`, `prop_borrowed`, `later`, `p_later`.

In [ ]:
# ← your answer here: replace every ... with code
n_loma = 24                        # from part 1
sample_loma = loma[(...) & (...)]  # Loma Prieta's sample again: days at least 1, and days below 10

rng = ...                 # a random-number generator, started from the seed 1989
own = []
borrowed = []
for run in ...:           # 2,000 times
    own.append(...)       # draw n_loma sizes from Loma Prieta's sample, and count those of magnitude 4 or more
    borrowed.append(...)  # the same, drawn from Ridgecrest's sample
own = np.array(own)
borrowed = np.array(borrowed)

prop_own = (...).mean()       # the proportion of the own weeks with 2 or more
prop_borrowed = (...).mean()  # the proportion of the borrowed weeks with 2 or more
later = loma[...]             # the rows from day 17 on
p_later = (...).mean()        # their proportion of magnitude 4 or more

print(f"weeks with 2 or more: {round(100 * prop_own)}% with Loma Prieta's own sample, {round(100 * prop_borrowed)}% with Ridgecrest's")
print((later["mag"] >= 4).sum(), "of", len(later), "earthquakes after day 17 were magnitude 4 or more: a proportion of", round(p_later, 3))

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert len(own) == 2000 and len(borrowed) == 2000, "own and borrowed each hold one number for every one of the 2000 weeks"
assert len(sample_loma) == ((loma["days"] >= 1) & (loma["days"] < 10)).sum() and (sample_loma["days"] >= 1).all(), "sample_loma keeps every row of loma from 1 day to 10 days after the mainshock, as in part 1"
assert 2.48 < own.mean() < 2.9 and 0.4 < borrowed.mean() < 0.58, "own draws n_loma sizes from sample_loma and keeps the COUNT of magnitude 4 or more, about 2.7 on average; borrowed draws them from sample, about 0.5"
assert abs(prop_own - (own >= 2).mean()) < 1e-9 and abs(prop_borrowed - (borrowed >= 2).mean()) < 1e-9, "each proportion is the proportion of weeks with 2 or more: the mean of a mask on own, and on borrowed"
assert (later["days"] >= 17).all() and len(later) == (loma["days"] >= 17).sum() and abs(p_later - (later["mag"] >= 4).mean()) < 1e-9, "later keeps the rows of loma from day 17 on, and p_later is its proportion of magnitude 4 or more"
print(f"✓ Homework 3 — 2 or more in {round(100 * prop_own)}% of the own weeks and {round(100 * prop_borrowed)}% of the borrowed ones; the later proportion was {round(p_later, 3)}")

**What the numbers mean.** Look at your two proportions. Under which forecast was "2 or more" an ordinary week, and
under which an unusual one? Does an unusual week prove a forecast wrong? Then compare the proportion
you measured from the later earthquakes with the two proportions of part 2. Where does it fall? Loma Prieta's own proportion rested on 12 earthquakes; think about how far a proportion
resting on so few could be from the truth.